In [0]:
%run ../../00_common/data_utils

In [0]:
def load_export_log(target_date):
    """
    从 golden_cdp_anonymization_database 读取 t_consumer_deletion_export_log，
    过滤 ExportDate = target_date 且 ExportType = ANORMALIZATION_GRACE_DATE 的数据。
    """
    anonymization_db = get_env_config('golden_cdp_anonymization_database')
    export_log_table = f"{anonymization_db}.t_consumer_deletion_export_log"

    export_log_df = spark.table(export_log_table).where(
        (F.col("ExportDate") == F.lit(target_date)) &
        (F.upper(F.col("ExportType")) == "ANORMALIZATION_GRACE_DATE")
    )

    return export_log_df

In [0]:
def match_with_master_consumer(exploded_df):
    """
    对同一个 market + brand + ukey, 
    比较 export_log 与 master_consumer 的(SourceSystemCode, ConsumerId) 组合集合是否完全一致，标记 is_matched。
    """
    golden_db = get_env_config('golden_consumer_master_database')
    master_consumer_table = f"{golden_db}.t_master_consumer"
    master_consumer_df = spark.table(master_consumer_table)

    export_keys_df = (
        exploded_df
        .groupBy("MarketCode", "BrandCode", "Ukey")
        .agg(
            F.collect_set(
                F.struct(
                    F.col("SourceSystemCode").alias("SourceSystemCode"),
                    F.col("ConsumerId").alias("ConsumerId")
                )
            ).alias("export_keys")
        )
    )

    master_keys_df = (
        master_consumer_df
        .groupBy(
            F.col("scon_mrkt_code").alias("MarketCode"),
            F.col("scon_brnd_code").alias("BrandCode"),
            F.col("consumermdmkey").alias("Ukey")
        )
        .agg(
            F.collect_set(
                F.struct(
                    F.col("scon_srcs_code").alias("SourceSystemCode"),
                    F.col("scon_consumerid").alias("ConsumerId")
                )
            ).alias("master_keys")
        )
    )

    group_match_df = (
        export_keys_df
        .join(master_keys_df, ["MarketCode", "BrandCode", "Ukey"], "left")
        .withColumn(
            "is_matched",
            F.col("master_keys").isNotNull() &
            (F.size(F.array_except(F.col("export_keys"), F.col("master_keys"))) == 0) &
            (F.size(F.array_except(F.col("master_keys"), F.col("export_keys"))) == 0)
        )
        .select("MarketCode", "BrandCode", "Ukey", "is_matched")
    )

    return (
        exploded_df
        .join(group_match_df, ["MarketCode", "BrandCode", "Ukey"], "left")
        .withColumn(
            "is_matched",
            F.coalesce(F.col("is_matched"), F.lit(False))
        )
    )

In [0]:
def determine_ukey_strategy(matched_df):
    """
    判断 4.1 / 4.2 两种匹配方式：
      4.1: 同一 market + ukey 下存在其他 brand 未在本次 export log 中 → 生成新 UUID
      4.2: 同一 market + ukey 下所有 brand 都在本次 export log 中 → 保持原 Ukey
    """
    golden_db = get_env_config('golden_consumer_master_database')
    master_consumer_df = spark.table(f"{golden_db}.t_master_consumer")

    # 按 market + ukey 聚合 master 中的所有 brand
    master_brands_df = (
        master_consumer_df
        .groupBy("scon_mrkt_code", "consumermdmkey")
        .agg(F.collect_set("scon_brnd_code").alias("master_brands"))
    )

    # 按 market + ukey 聚合本次 export log 中匹配到的 brand
    export_brands_df = (
        matched_df
        .filter(F.col("is_matched"))
        .groupBy("MarketCode", "Ukey")
        .agg(F.collect_set("BrandCode").alias("export_brands"))
    )

    matched_with_brands = (
        matched_df
        .join(
            master_brands_df,
            (F.col("MarketCode") == F.col("scon_mrkt_code")) &
            (F.col("Ukey") == F.col("consumermdmkey")),
            "left"
        )
        .join(export_brands_df, ["MarketCode", "Ukey"], "left")
        .withColumn(
            "has_other_brands",
            F.when(
                F.col("master_brands").isNotNull() & F.col("export_brands").isNotNull(),
                F.size(F.array_except(F.col("master_brands"), F.col("export_brands"))) > 0
            ).otherwise(F.lit(False))
        )
    )

    # 对 case 4.1 的每个 (market, ukey) 生成同一个 UUID（同 ukey 的所有 brand 共享）
    new_ukey_df = (
        matched_with_brands
        .filter(F.col("is_matched") & F.col("has_other_brands"))
        .select("MarketCode", "Ukey")
        .distinct()
        .withColumn("New_UniversalKey", F.expr("uuid()"))
    )

    result_df = (
        matched_with_brands
        .join(new_ukey_df, ["MarketCode", "Ukey"], "left")
        .withColumn(
            "New_UniversalKey",
            F.coalesce(F.col("New_UniversalKey"), F.col("Ukey"))
        )
        .withColumn(
            "active_type",
            F.when(F.col("is_matched"), F.lit(ANON_ACTIVE_TYPE_ANONYMIZATION)).otherwise(F.lit(ANON_ACTIVE_TYPE_NOT_MATCH))
        )
    )

    return result_df

In [0]:
def write_anonymization_log(result_df, task_id):
    """
    将处理结果追加写入 t_mdm_anonymization_log。
    """
    anonymization_db = get_env_config('silver_mdm_anonymization_database')
    log_table = f"{anonymization_db}.t_mdm_anonymization_log"

    log_df = result_df.select(
        F.col("ConsumerId"),
        F.col("MarketCode"),
        F.col("BrandCode"),
        F.col("SourceSystemCode"),
        F.col("Ukey").alias("Old_UniversalKey"),
        F.col("New_UniversalKey"),
        F.col("ExportDate"),
        F.col("active_type"),
        F.lit("").alias("comment"),
        F.current_timestamp().alias("create_time"),
        F.lit(task_id).alias("task_id"),
        F.expr("uuid()").alias("cal_uuid")
    )

    save_to_target_table(log_df, log_table, f"task_id='{task_id}'")

In [0]:
def update_master_consumer_table(anonymization_df, task_id):
    """
    更新 t_master_consumer：
      - consumermdmkey = New_UniversalKey
      - scon_srcc_action = 'DELETE'
      - 按 5.1_generate_master_consumer_tables.py 的 DELETE 逻辑清空 PII 字段
      - task_id = 当前 task_id
    按 BusinessKey (market, brand, sourcesystemcode, consumerid) 匹配。
    """
    golden_db = get_env_config('golden_consumer_master_database')
    master_consumer_table = f"{golden_db}.t_master_consumer"

    source_df = anonymization_df.select(
        F.col("MarketCode"),
        F.col("BrandCode"),
        F.col("SourceSystemCode"),
        F.col("ConsumerId"),
        F.col("New_UniversalKey")
    ).distinct()

    if source_df.isEmpty():
        print("No matched anonymization records, skip master consumer update")
        return

    master_consumer_delta = DeltaTable.forName(spark, master_consumer_table)
    (
        master_consumer_delta.alias("target")
        .merge(
            source_df.alias("source"),
            """
            target.scon_srcs_code = source.SourceSystemCode AND
            target.scon_mrkt_code = source.MarketCode AND
            target.scon_brnd_code = source.BrandCode AND
            target.scon_consumerid = source.ConsumerId
            """
        )
        .whenMatchedUpdate(set={
            "consumermdmkey": F.col("source.New_UniversalKey"),
            "scon_srcc_action": F.lit("DELETE"),
            "scon_salutation": F.lit(""),
            "scon_englishfirstname": F.lit(""),
            "scon_englishmiddlename": F.lit(""),
            "scon_englishlastname": F.lit(""),
            "scon_englishfullname": F.lit(""),
            "scon_localfirstname": F.lit(""),
            "scon_localmiddlename": F.lit(""),
            "scon_locallastname": F.lit(""),
            "scon_localfullname": F.lit(""),
            "scon_localfirstname2": F.lit(""),
            "scon_localmiddlename2": F.lit(""),
            "scon_locallastname2": F.lit(""),
            "scon_localfullname2": F.lit(""),
            "scon_identitynum": F.lit(""),
            "scon_passportnum": F.lit(""),
            "scon_socialsecuritynum": F.lit(""),
            "scon_birthday": F.lit(None),
            "task_id": F.lit(task_id),
            "scon_update_dt": F.current_timestamp(),
            "scon_update_uid": F.lit("ELC")
        })
        .execute()
    )

In [0]:
def _build_pii_update_df(master_df, base_df, id_col, mrkt_col, fk_col, filter_expr=None):
    """
    构造单个 PII 表的待更新数据集：按 (id, market) 聚合最大 delete_timestamp。
    通过 fk_col（如 scme_scon_id）关联 master_consumer 的 scon_id。
    """
    df = (
        master_df.alias("m")
        .join(
            base_df.alias("base"),
            (F.col(f"m.{fk_col}") == F.col("base.scon_id")) &
            (F.col(f"m.{mrkt_col}") == F.col("base.scon_mrkt_code")),
            "inner"
        )
    )
    if filter_expr is not None:
        df = df.where(filter_expr)
    return (
        df.groupBy(f"m.{id_col}", f"m.{mrkt_col}")
        .agg(F.max("base.delete_timestamp").alias("delete_timestamp"))
        .cache()
    )


def _clear_pii_table(master_df, base_df, table_name, id_col, mrkt_col, fk_col, filter_expr, update_set):
    """
    对单个 PII 表执行 Delta merge update，返回更新行数。
    """
    to_update = _build_pii_update_df(master_df, base_df, id_col, mrkt_col, fk_col, filter_expr)

    update_count = to_update.count()
    if update_count > 0:
        (
            DeltaTable.forName(spark, table_name).alias("target")
            .merge(
                to_update.alias("source"),
                f"target.{id_col} = source.{id_col} AND target.{mrkt_col} = source.{mrkt_col}"
            )
            .whenMatchedUpdate(set=update_set)
            .execute()
        )

    to_update.unpersist()
    return update_count

def update_master_other_tables(anonymization_df):
    """
    清空 t_master_phone / t_master_emedia / t_master_address / t_master_optin 中的 PII。
    参考 05_Master_Data_Generate/5.1_generate_master_consumer_tables.py 的 DELETE 逻辑，
    使用 current_timestamp() 作为 source timestamp。
    """
    golden_db = get_env_config('golden_consumer_master_database')
    master_consumer_table = f"{golden_db}.t_master_consumer"
    master_emedia_table = f"{golden_db}.t_master_emedia"
    master_phone_table = f"{golden_db}.t_master_phone"
    master_address_table = f"{golden_db}.t_master_address"
    master_optin_table = f"{golden_db}.t_master_optin"

    master_consumer_df = spark.table(master_consumer_table)
    master_emedia_df = spark.table(master_emedia_table)
    master_phone_df = spark.table(master_phone_table)
    master_address_df = spark.table(master_address_table)
    master_optin_df = spark.table(master_optin_table)

    # 构造 DELETE 基础数据集（scon_id + market + 当前时间戳）
    delete_consumer_base_df = (
        master_consumer_df.alias("scon")
        .join(
            anonymization_df.alias("src"),
            (F.col("scon.scon_srcs_code") == F.col("src.SourceSystemCode")) &
            (F.col("scon.scon_mrkt_code") == F.col("src.MarketCode")) &
            (F.col("scon.scon_brnd_code") == F.col("src.BrandCode")) &
            (F.col("scon.scon_consumerid") == F.col("src.ConsumerId")),
            "inner"
        )
        .select(
            F.col("scon.scon_id"),
            F.col("scon.scon_mrkt_code"),
            F.current_timestamp().alias("delete_timestamp")
        )
        .cache()
    )

    if delete_consumer_base_df.isEmpty():
        print("clear-PII completed: emedia=0, phone=0, address=0, optin=0")
        delete_consumer_base_df.unpersist()
        return

    # 1) emedia: 清空 scme_address
    emedia_update_count = _clear_pii_table(
        master_emedia_df, delete_consumer_base_df, master_emedia_table,
        "scme_id", "scme_mrkt_code", "scme_scon_id",
        F.coalesce(F.col("m.scme_address"), F.lit("")) != "",
        {
            "scme_address": F.lit(""),
            "scme_sourcetimestamp": F.col("source.delete_timestamp"),
            "scme_update_dt": F.current_timestamp(),
            "scme_update_uid": F.lit("ELC")
        }
    )

    # 2) phone: 清空 scph_phonenumber
    phone_update_count = _clear_pii_table(
        master_phone_df, delete_consumer_base_df, master_phone_table,
        "scph_id", "scph_mrkt_code", "scph_scon_id",
        F.coalesce(F.col("m.scph_phonenumber"), F.lit("")) != "",
        {
            "scph_phonenumber": F.lit(""),
            "scph_sourcetimestamp": F.col("source.delete_timestamp"),
            "scph_update_dt": F.current_timestamp(),
            "scph_update_uid": F.lit("ELC")
        }
    )

    # 3) address: 清空 scad_address1/2/3
    address_update_count = _clear_pii_table(
        master_address_df, delete_consumer_base_df, master_address_table,
        "scad_id", "scad_mrkt_code", "scad_scon_id",
        F.concat_ws("", F.col("m.scad_address1"), F.col("m.scad_address2"), F.col("m.scad_address3")) != "",
        {
            "scad_address1": F.lit(""),
            "scad_address2": F.lit(""),
            "scad_address3": F.lit(""),
            "scad_sourcetimestamp": F.col("source.delete_timestamp"),
            "scad_update_dt": F.current_timestamp(),
            "scad_update_uid": F.lit("ELC")
        }
    )

    # 4) optin: 将 scop_optin_flag 置 0
    optin_update_count = _clear_pii_table(
        master_optin_df, delete_consumer_base_df, master_optin_table,
        "scop_id", "scop_mrkt_code", "scop_scon_id",
        F.col("m.scop_optin_flag").isNotNull() & (F.col("m.scop_optin_flag") != F.lit(False)),
        {
            "scop_optin_flag": F.lit(False),
            "scop_optin_dt": F.col("source.delete_timestamp"),
            "scop_update_dt": F.current_timestamp(),
            "scop_update_uid": F.lit("ELC")
        }
    )

    print(
        f"clear-PII completed: emedia={emedia_update_count}, "
        f"phone={phone_update_count}, address={address_update_count}, "
        f"optin={optin_update_count}"
    )

    delete_consumer_base_df.unpersist()

In [0]:
def anonymize_master_data(anonymization_df, task_id):
    """
    整合 #6.1 + #6.3: 
      - 更新 t_master_consumer (key/action/PII/task_id)
      - 清空 t_master_emedia / phone / address / optin 中的 PII
    """
    update_master_consumer_table(anonymization_df, task_id)
    update_master_other_tables(anonymization_df)

In [0]:
def update_other_brands_task_id(active_df, task_id):
    """
    对于 4.1 方式生成新 Ukey 的记录，
    将同一 market + ukey 下的其他 brand 的 task_id 更新为当前 task_id。
    """
    golden_db = get_env_config('golden_consumer_master_database')
    master_consumer_table = f"{golden_db}.t_master_consumer"
    master_consumer_df = spark.table(master_consumer_table)

    other_brands_df = (
        master_consumer_df.alias("mc")
        .join(
            active_df.alias("src"),
            (F.col("mc.scon_mrkt_code") == F.col("src.MarketCode")) &
            (F.col("mc.consumermdmkey") == F.col("src.Ukey")) &
            (F.col("mc.scon_brnd_code") != F.col("src.BrandCode")),
            "inner"
        )
        .select(
            F.col("mc.scon_id").alias("scon_id"),
            F.col("mc.scon_mrkt_code").alias("scon_mrkt_code")
        )
        .distinct()
    )

    if other_brands_df.isEmpty():
        print("No other brands to update task_id")
        return

    master_consumer_delta = DeltaTable.forName(spark, master_consumer_table)
    (
        master_consumer_delta.alias("target")
        .merge(
            other_brands_df.alias("source"),
            """
            target.scon_id = source.scon_id AND
            target.scon_mrkt_code = source.scon_mrkt_code
            """
        ).whenMatchedUpdate(set={
            "task_id": F.lit(task_id),
            "scon_update_dt": F.current_timestamp(),
            "scon_update_uid": F.lit("ELC")
        })
        .execute()
    )

In [0]:
def delete_old_derived_records(active_df):
    """
    S4: 对于 4.1 方式生成新 Ukey 的记录，
    删除 t_derived_consumer_l2 / t_derived_consumer_l3 中残留的
    market + old_ukey(consumermdmkey) + brand 数据。
    """
    if active_df.isEmpty():
        print("No active records, skip derived table deletion")
        return

    golden_db = get_env_config('golden_consumer_master_database')
    l2_table = f"{golden_db}.t_derived_consumer_l2"
    l3_table = f"{golden_db}.t_derived_consumer_l3"

    delete_keys_df = (
        active_df
        .select(
            F.col("MarketCode").alias("scon_mrkt_code"),
            F.col("Ukey").alias("consumermdmkey"),
            F.col("BrandCode").alias("scon_brnd_code")
        )
    )

    for table_name in [l2_table, l3_table]:
        (
            DeltaTable.forName(spark, table_name).alias("target")
            .merge(
                delete_keys_df.alias("source"),
                """
                target.scon_mrkt_code = source.scon_mrkt_code AND
                target.consumermdmkey = source.consumermdmkey AND
                target.scon_brnd_code = source.scon_brnd_code
                """
            )
            .whenMatchedDelete()
            .execute()
        )

In [0]:
def handle_profile(task_id):
    """
    Consumer Deletion / Anonymization 主流程。
    """
    target_date_str = spark.sql("SELECT date_sub(current_date, 1) as d").collect()[0]["d"]
    print(f"Processing ExportDate: {target_date_str}")

    print("1. load and explode DA export log")

    # 1. 读取并过滤 export_log
    export_log_df = load_export_log(target_date_str)

    if export_log_df.isEmpty():
        print("No export_log records for target date, nothing to do")
        return

    # 2. 展开 ConsumerIdList
    # MAP<STRING,STRING>格式: key=ConsumerId, value=SourceSystemCode
    exploded_df = (
        export_log_df
        .select(
            F.col("Ukey"),
            F.col("MarketCode"),
            F.col("BrandCode"),
            F.explode_outer(F.col("ConsumerIdList")).alias("ConsumerId", "SourceSystemCode"),
            F.col("ExportDate")
        )
       .where(
            F.col("Ukey").isNotNull() &
            F.col("MarketCode").isNotNull() &
            F.col("BrandCode").isNotNull() &
            F.col("ConsumerId").isNotNull() &
            F.col("SourceSystemCode").isNotNull()
        )
    )

    print("2. match and generate new ukey")

    # 3. 与 master_consumer 匹配
    matched_df = match_with_master_consumer(exploded_df)

    # 4. 判断 4.1 / 4.2 并生成 New_UniversalKey
    result_df = determine_ukey_strategy(matched_df)

    result_df = result_df.checkpoint(eager=True)
    result_count = result_df.count()
    print(f'total count: {result_count}')
    
    print("3. merge consumer anonymization log")

    # 5. 写入 t_mdm_anonymization_log
    write_anonymization_log(result_df, task_id)

    print("4. update master table & clear pii for matched anonymization consumer")

    # 6. 对匹配的数据进行更新
    anonymization_df = result_df.filter(F.col("active_type") == ANON_ACTIVE_TYPE_ANONYMIZATION)

    anonymization_df = anonymization_df.checkpoint(eager=True)
    anonymization_count = anonymization_df.count()

    if anonymization_count == 0:
        print("No matched anonymization records, skip downstream updates")
        return

    print(f'matched anonymization count: {anonymization_count}')

    # 6.1 更新master表并清空 PII
    anonymize_master_data(anonymization_df, task_id)

    print("5. update task_id for other brand keys")

    # 6.2 更新同 market+ukey 下其他 brand 的 task_id
    active_df = (
        anonymization_df
        .filter(F.col("has_other_brands"))
        .select("MarketCode", "BrandCode", "Ukey")
        .distinct()    
    )

    active_df = active_df.checkpoint(eager=True)
    active_count = active_df.count()
    print(f'keys count: {active_count}')

    update_other_brands_task_id(active_df, task_id)

    print("6. delete old derived data for anonymized keys")
    print(f'keys count: {active_count}')

    # 6.3 删除 derived l2/l3 中残留的老 ukey + brand 数据
    delete_old_derived_records(active_df)

In [0]:
task_id = dbutils.widgets.get("task_id")
print(f"task_id: {task_id}")

step_name = "handle_profile"
step_num = "01"
project = "dataanonymization"
log_table_name = f"{get_env_config('config_database')}.t_task_step_log"

start_time = datetime.now()
status = "SUCCESS"
message = "completed"

try:
    spark.sparkContext.setCheckpointDir(f"{get_env_config('checkpoint_path_consumer_master')}/{task_id}")
    handle_profile(task_id)
except Exception as e:
    status = "FAILED"
    message = f"{type(e).__name__}: {str(e)}"
    raise
finally:
    end_time = datetime.now()
    append_step_log(
        log_table_name=log_table_name,
        task_id=task_id,
        step_num=step_num,
        step_name=step_name,
        start_time=start_time,
        end_time=end_time,
        status=status,
        message=message,
        project=project
    )